In [ ]:
import sys
!{sys.executable} -m pip install seaborn


In [ ]:
import sys; print(sys.path)


['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/usr/local/lib/python3.12/dist-packages/setuptools/_vendor']


In [ ]:
!pip install japanize-matplotlib
import os
import sys
import yaml
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import shap
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
import japanize_matplotlib #日本語表示対応


# Notebook から src ディレクトリを追加
# sys.path.append("../src")



In [ ]:
import warnings
warnings.filterwarnings(
    "ignore",
    category=pd.errors.PerformanceWarning
)

## 01.config読み込み

In [ ]:
!git clone https://github.com/keiseki-eng/My_Python_project.git
conf_path = os.path.join('My_Python_project//config/含水率判定/config.yaml')
with open(conf_path, 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
# 定義した特徴量リストを読み込み
feature_list = config['FEATURE']['FEATURE_LIST']

In [ ]:
# カテゴリカルカラムのリストを定義
categorical_cols = config['FEATURE']['CATEGORICAL_COLS']

## 02.データ読み込み

In [ ]:
!git clone https://github.com/keiseki-eng/My_Python_project.git
df_train = pd.read_csv(
    "My_Python_project/20.Data/含水率判定/train.csv",
    encoding="cp932"
)

df_test = pd.read_csv(
    "My_Python_project/20.Data/含水率判定/test.csv",
    encoding="cp932"
)



## 03.前処理

In [ ]:
# df_trainのカテゴリカルカラムを'category'型に変換
for col in categorical_cols:
    if col in df_train.columns:
        df_train[col] = df_train[col].astype('category')

# df_testのカテゴリカルカラムを'category'型に変換
for col in categorical_cols:
    if col in df_test.columns:
        df_test[col] = df_test[col].astype('category')

In [ ]:
# # 科目特徴量を追加
# train_conditions = [
#     df_train["樹種"] == "イチョウ",
#     df_train["樹種"] == "ウエンジ",
#     df_train["樹種"] == "ウォールナット",
#     df_train["樹種"] == "クリ",
#     df_train["樹種"] == "スプルース",
#     df_train["樹種"] == "チェリー",
#     df_train["樹種"] == "トチ",
#     df_train["樹種"] == "ナラ",
#     df_train["樹種"] == "ヒノキ",
#     df_train["樹種"] ==  "ベイスギ",
#     df_train["樹種"] ==  "米ヒバ",
#     df_train["樹種"] == "ベイマツ",
#     df_train["樹種"] == "ホワイトオーク",
# ]
# test_conditions = [
#     df_test["樹種"] == "クスノキ",
#     df_test["樹種"] == "ケヤキ",
#     df_test["樹種"] == "スギ",
#     df_test["樹種"] == "タモ",
#     df_test["樹種"] == "チーク",
#     df_test["樹種"] == "ヤマザクラ",
# ]
# train_choices = [
#     "イチョウ",
#     "マメ",
#     "クルミ",
#     "ブナ",
#     "マツ",
#     "バラ",
#     "ムクロジ",
#     "ブナ",
#     "ヒノキ",
#     "ヒノキ",
#     "ヒノキ",
#     "マツ",
#     "ブナ",
# ]
# test_choices = [
#     "クスノキ",
#     "ニレ",
#     "ヒノキ",
#     "モクセイ",
#     "クマツヅラ",
#     "バラ",
# ]



# df_train["科目"] = np.select(
#     train_conditions,
#     train_choices,
#     default = ""
# )
# df_test["科目"] = np.select(
#     test_conditions,
#     test_choices,
#     default = ""
# )

In [ ]:
# # 科目特徴量を追加
# train_conditions = [
#     df_train["樹種"] == "イチョウ",
#     df_train["樹種"] == "ウエンジ",
#     df_train["樹種"] == "ウォールナット",
#     df_train["樹種"] == "クリ",
#     df_train["樹種"] == "スプルース",
#     df_train["樹種"] == "チェリー",
#     df_train["樹種"] == "トチ",
#     df_train["樹種"] == "ナラ",
#     df_train["樹種"] == "ヒノキ",
#     df_train["樹種"] ==  "ベイスギ",
#     df_train["樹種"] ==  "米ヒバ",
#     df_train["樹種"] == "ベイマツ",
#     df_train["樹種"] == "ホワイトオーク",
# ]
# test_conditions = [
#     df_test["樹種"] == "クスノキ",
#     df_test["樹種"] == "ケヤキ",
#     df_test["樹種"] == "スギ",
#     df_test["樹種"] == "タモ",
#     df_test["樹種"] == "チーク",
#     df_test["樹種"] == "ヤマザクラ",
# ]
# train_choices = [
#     0.55,
#     0.90,
#     0.64,
#     0.56,
#     0.45,
#     0.50,
#     0.52,
#     0.68,
#     0.41,
#     0.37,
#     0.40,
#     0.53,
#     0.75,
# ]
# test_choices = [
#     0.52,  # クスノキ
#     0.69,  # ケヤキ
#     0.38,  # スギ
#     0.63,  # タモ
#     0.66,  # チーク
#     0.60,  # ヤマザクラ

# ]



# df_train["密度"] = np.select(
#     train_conditions,
#     train_choices,
#     default = np.nan
# )
# df_test["密度"] = np.select(
#     test_conditions,
#     test_choices,
#     default = np.nan
# )

In [ ]:
# softwood = ["ヒノキ","スギ","ベイスギ","米ヒバ","ベイマツ","スプルース"]

# df_train["針葉樹"] = df_train["樹種"].isin(softwood).astype(int)
# df_test["針葉樹"] = df_test["樹種"].isin(softwood).astype(int)

In [ ]:
# # 欠損値はカラムの型に応じて補完
# for col in df_train.columns:
#     if col == "money_room": # 目的変数はこの時点では補完しない
#         continue

#     # if col in categorical_cols: # 明示的に定義されたカテゴリカルカラム
#     #     # LightGBMが0をカテゴリとして扱えるため、0で補完
#     #     df_train[col] = df_train[col].fillna(0)
#     elif df_train[col].dtype in ['int64', 'float64']: # 数値カラム
#         # 数値カラムは訓練データの中央値で補完
#         median_val = df_train[col].median()
#         df_train[col] = df_train[col].fillna(median_val)
#         df_test[col] = df_test[col].fillna(median_val)
#     # else:
#         # # その他の型（objectなど）は、とりあえず0で補完（必要に応じて調整）
#         # df_train[col] = df_train[col].fillna(0)


In [ ]:
# # # room_count
# # # unit_area
# df_train = df_train[(df_train["unit_area"]>12.5) & (df_train["unit_area"]<300)]

##### 日付関係のデータは経過日数を特徴量とする

In [ ]:
# # 'target_ym'をdatetime型に変換 (月の最初の日と仮定)
# df_train['target_date'] = pd.to_datetime(df_train['target_ym'].astype(str) + '01', format='%Y%m%d')
# df_test['target_date'] = pd.to_datetime(df_test['target_ym'].astype(str) + '01', format='%Y%m%d')

# # 'building_create_date'をdatetime型に変換
# df_train['building_create_date'] = pd.to_datetime(df_train['building_create_date'])
# df_test['building_create_date'] = pd.to_datetime(df_test['building_create_date'])
# # 築年数を計算 (年単位)
# df_train['building_age'] = (df_train['target_date'] - df_train['building_create_date']).dt.days // 365
# df_test['building_age'] = (df_test['target_date'] - df_test['building_create_date']).dt.days // 365

# # 'renovation_date'がUNIXタイムスタンプになっているため、再度datetimeに変換して経過年数を計算
# # 既に'renovation_date'はUNIXタイムスタンプに変換済みのため、元の日付情報に戻すか、元のカラムを使用する
# # ここでは、元のカラムが保持されていると仮定し、新しいカラムとして計算
# # 注: もしdf_train['renovation_date']がUNIXタイムスタンプのみになっている場合、この処理は調整が必要です。
# # 既存のrenovation_date（UNIXタイムスタンプ）を無視し、元の文字列カラムが存在しないため、新たに変換する
# # もし元の文字列カラムが失われている場合は、UNIXタイムスタンプをdatetimeに変換する
# # 現状のNotebookの処理から判断すると、renovation_dateはUNIXタイムスタンプなので、それを基に経過年数を計算します。

# # UNIXタイムスタンプからdatetimeへの変換
# df_train['renovation_datetime'] = pd.to_datetime(df_train['renovation_date'], errors='coerce')
# df_test['renovation_datetime'] = pd.to_datetime(df_test['renovation_date'], errors='coerce')

# # リノベーションからの経過年数を計算 (年単位)
# # リノベーション日がない場合はNaNとなり、NaNとtarget_dateの差もNaNになるため、後でfillnaで処理
# df_train['years_since_renovation'] = (df_train['target_date'] - df_train['renovation_datetime']).dt.days // 365
# df_test['years_since_renovation'] = (df_test['target_date'] - df_test['renovation_datetime']).dt.days // 365
# # 欠損値の補完
# # building_ageとyears_since_renovationの負の値（未来の日付）やNaNを処理
# # 例: 負の値やNaNを0で補完する
# df_train['building_age'] = df_train['building_age'].apply(lambda x: max(0, x) if pd.notna(x) else 0)
# df_test['building_age'] = df_test['building_age'].apply(lambda x: max(0, x) if pd.notna(x) else 0)

# df_train['years_since_renovation'] = df_train['years_since_renovation'].apply(lambda x: max(0, x) if pd.notna(x) else 0)
# df_test['years_since_renovation'] = df_test['years_since_renovation'].apply(lambda x: max(0, x) if pd.notna(x) else 0)
# # 新しい特徴量をfeature_listに追加
# new_features = ['building_age', 'years_since_renovation']
# feature_list.extend(new_features)

# # feature_listから'target_date', 'building_create_date', 'renovation_datetime'を削除 (直接特徴量として使わないため)
# # (元のbuilding_create_dateとrenovation_dateはUNIX timestampとして残しておく)
# feature_list = [f for f in feature_list if f not in ['target_date', 'building_create_date', 'renovation_date', 'renovation_datetime']]

# # 確認
# # print(df_train[feature_list].head())



## 04.特徴量生成

In [ ]:
# # addr1_1 と addr2_1をあわせた特徴量を作成
# df_train['addr1+2'] = df_train['addr1_1'].astype(str) + '_' + df_train['addr1_2'].astype(str)
# df_test['addr1+2'] = df_test['addr1_1'].astype(str) + '_' + df_test['addr1_2'].astype(str)
# feature_list.append('addr1+2')

In [ ]:
# # 極端な値はクリッピング
# for col in feature_list:
#     # 数値型でなければスキップ
#     if not pd.api.types.is_numeric_dtype(df_train[col]):
#         continue

#     # 全部NaNの列はスキップ
#     if df_train[col].notna().sum() == 0:
#         continue

#     # 1% / 99% パーセンタイル
#     lower = df_train[col].quantile(0.01)
#     upper = df_train[col].quantile(0.99)

#     # パーセンタイルが計算できない場合はスキップ
#     if pd.isna(lower) or pd.isna(upper):
#         continue

#     # クリッピング
#     df_train[col] = df_train[col].clip(lower, upper)


In [ ]:
# LightGBM 用に使用するカテゴリ特徴量のリストを作成
categorical_features = [
    col for col in categorical_cols
    if col in feature_list
]

feature_list = feature_list.copy()

In [ ]:
# df_train['addr1+2'] = df_train['addr1+2'].astype('category')
# df_test['addr1+2']  = df_test['addr1+2'].astype('category')


＿前処理後のDataFrameを出力保存

In [ ]:
# 前処理後のDataFrameをpklファイルで出力保存
# ★Colabローカル環境に保存されるので、githubに移管要！

df_train.to_pickle('My_Python_project/20.Data/含水率判定/preprocessed_train.pkl')
df_test.to_pickle('My_Python_project/20.Data/含水率判定/preprocessed_test.pkl')